# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [2]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --
from surprise import accuracy
from surprise import model_selection
from surprise import Reader
from surprise import Dataset
from surprise import SVD
 
# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
from models import get_top_n
# -- add new imports here --

# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [10]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    trainset, testset = model_selection.train_test_split(ratings_dataset, test_size=eval_config.test_size)
    algo.fit(trainset)
    predictions = algo.test(testset)
    return predictions


def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    # -- implement the function generate_loo_top_n --
    loo = model_selection.LeaveOneOut(n_splits=1, random_state=1)
    trainset, testset = next(loo.split(ratings_dataset))
    algo.fit(trainset)
    anti_test_set = trainset.build_anti_testset()
    predictions = algo.test(anti_test_set)
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    

    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user with full training set (LOO)"""
    # -- implement the function generate_full_top_n --
    full_train_set = ratings_dataset.build_full_trainset()
    algo.fit(full_train_set)
    anti_test_set = full_train_set.build_anti_testset()
    predictions = algo.test(anti_test_set)
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n


def precompute_information():
    """ Returns a dictionary that precomputes relevant information for evaluating in full mode
    
    Dictionary keys:
    - precomputed_dict["item_to_rank"] : contains a dictionary mapping movie ids to rankings
    - (-- for your project, add other relevant information here -- )
    """
    df_ratings = load_ratings()
    #group by item id and count the number of ratings
    df_ratings = df_ratings.groupby(C.ITEM_ID_COL).count().reset_index()
    df_ratings = df_ratings.sort_values(by=C.RATING_COL, ascending=False)
    #create a diconary mapping movie ids to rankings
    item_to_rank = {}
    for rank, row in enumerate(df_ratings.itertuples(index=False), start=1):
        item_id = getattr(row, C.ITEM_ID_COL)
        item_to_rank[item_id] = rank


    
    precomputed_dict = {}
    precomputed_dict["item_to_rank"] = item_to_rank
    return precomputed_dict                


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    algo = SVD()
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [12]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric)
    
    A hit (1) happens when the movie in the testset has been picked by the top-n recommender
    A fail (0) happens when the movie in the testset has not been picked by the top-n recommender
    """
    # -- implement the function get_hit_rate --
    hits = 0
    total = 0
    
    # Boucle sur chaque tuple du testset
    for (user, left_out_movie, _) in testset:
        # Récupérer les recommandations pour ce user
        user_top_n = anti_testset_top_n.get(user, [])
        
        # Récupérer juste les IDs de film recommandés
        recommended_movies = [movie_id for (movie_id, _) in user_top_n]
        
        # Vérifier si le film "laissé de côté" est dans les recommandations
        if left_out_movie in recommended_movies:
            hits += 1
        
        total += 1
    hit_rate = hits / total if total > 0 else 0
    return hit_rate


def get_novelty(anti_testset_top_n, item_to_rank):
    """Compute the average novelty of the top-n recommendation over the users (full metric)
    
    The novelty is defined as the average ranking of the movies recommended
    """
    # -- implement the function get_novelty --
    total_rank_sum = 0
    total_items = 0
    rows = []
    for user_id, item_list in anti_testset_top_n.items():
        for item_id, estimated_rating in item_list:
            rows.append((user_id, item_id, estimated_rating))
    df_topn_full = pd.DataFrame(rows, columns=['user', 'item', 'estimated_rating'])
    for _, row in df_topn_full.iterrows():
        item_id = row['item']
        rank = item_to_rank.get(item_id, 0)  # 0 si l’item n’est pas dans item_to_rank
        total_rank_sum += rank
        total_items += 1
    average_rank_sum = total_rank_sum / total_items 
    
    return average_rank_sum

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [14]:
AVAILABLE_METRICS = {
    "split": {
        "mae": (accuracy.mae, {'verbose': False}),
        # -- add new split metrics here --
    "rmse" : (accuracy.rmse, {'verbose': False}),
    }
    # -- add new types of metrics here --
    ,"loo" : {"hit_rate" : (get_hit_rate, {})},
    "full" : {"novelty" : (get_novelty, {})}
}

sp_ratings = load_ratings(surprise_format=True)
algo = SVD()
test = generate_split_predictions(algo, sp_ratings, EvalConfig)

top_n_loo_top,test_set_loo = generate_loo_top_n(algo, sp_ratings, EvalConfig)
rows = []
for user_id, item_list in top_n_loo_top.items():
    for item_id, estimated_rating in item_list:
        rows.append((user_id, item_id, estimated_rating))

# Création du DataFrame
df_topn = pd.DataFrame(rows, columns=['user', 'item', 'estimated_rating'])
df_topn.to_csv("top_n_loo.csv", index=False)


# Affichage des 50 premières lignes du DataFrame



top_n_full = generate_full_top_n(algo, sp_ratings, EvalConfig) 
rows = []
for user_id, item_list in top_n_full.items():
    for item_id, estimated_rating in item_list:
        rows.append((user_id, item_id, estimated_rating))
df_topn_full = pd.DataFrame(rows, columns=['user', 'item', 'estimated_rating'])

df_topn_full.to_csv("top_n_full.csv", index=False)
precomputed_dict = precompute_information()
evaluation_report = create_evaluation_report(EvalConfig, sp_ratings, precomputed_dict, AVAILABLE_METRICS)
display(evaluation_report)
export_evaluation_report(evaluation_report)



Handling model baseline_1
Training split predictions
- computing metric mae
Training loo predictions
Training full predictions


,mae,hit_rate,novelty
baseline_1,1.659214,0.002981,4596.507899


In [7]:
df_ratings = load_ratings()
#group by item id and count the number of ratings
df_ratings = df_ratings.groupby(C.ITEM_ID_COL).count().reset_index()
df_ratings = df_ratings.sort_values(by=C.RATING_COL, ascending=False)
#create a diconary mapping movie ids to rankings
item_to_rank = {}
for rank, row in enumerate(df_ratings.itertuples(index=False), start=1):
    item_id = getattr(row, C.ITEM_ID_COL)
    item_to_rank[item_id] = rank

print(item_to_rank)


{356: 1, 296: 2, 318: 3, 593: 4, 260: 5, 480: 6, 2571: 7, 1: 8, 527: 9, 589: 10, 1196: 11, 110: 12, 1270: 13, 608: 14, 2858: 15, 1198: 16, 780: 17, 1210: 18, 588: 19, 457: 20, 590: 21, 2959: 22, 47: 23, 50: 24, 364: 25, 858: 26, 150: 27, 4993: 28, 380: 29, 592: 30, 32: 31, 2762: 32, 2028: 33, 1580: 34, 5952: 35, 377: 36, 595: 37, 7153: 38, 344: 39, 4306: 40, 648: 41, 1265: 42, 1721: 43, 1197: 44, 3578: 45, 1097: 46, 231: 47, 1240: 48, 1704: 49, 367: 50, 500: 51, 1036: 52, 736: 53, 1073: 54, 34: 55, 1291: 56, 2716: 57, 597: 58, 541: 59, 1136: 60, 316: 61, 1193: 62, 165: 63, 6539: 64, 2628: 65, 1682: 66, 733: 67, 1221: 68, 5349: 69, 2918: 70, 1089: 71, 293: 72, 4226: 73, 1213: 74, 4963: 75, 4886: 76, 586: 77, 153: 78, 1214: 79, 3793: 80, 587: 81, 2997: 82, 8961: 83, 539: 84, 1200: 85, 1617: 86, 253: 87, 3114: 88, 4973: 89, 778: 90, 1923: 91, 924: 92, 357: 93, 6377: 94, 10: 95, 2396: 96, 1206: 97, 58559: 98, 1732: 99, 39: 100, 1961: 101, 111: 102, 1527: 103, 912: 104, 919: 105, 1968: 106,